In [18]:
import pandas as pd
import os

# Define the path to the raw dataset
file_path = os.path.join("resources", "games.csv")

# Load the data
df_raw = pd.read_csv(file_path)

# Displaying headers and the first 5 rows
print("Detected Columns (Raw):", df_raw.columns.tolist())
display(df_raw.head())

Detected Columns (Raw): ['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


## The CSV Trap: Why Flat Files Failed

Initially, the project attempted to source data from `games.csv`. However, this approach revealed significant **data integrity issues** that are common when dealing with scraped web data:

* **Delimiter Conflicts:** Many game descriptions contain unquoted commas, which the CSV parser misinterprets as new column boundaries.
* **Column Shifting:** These extra delimiters caused a "ripple effect," shifting critical data (like release dates and review counts) into the wrong columns.
* **Structural Mismatch:** The header row contained 39 definitions, while many data rows contained 42+ values, making automated parsing unreliable.

To ensure **100% data accuracy** for this analysis, I have pivoted to using `games.json`. Unlike CSV, the JSON format uses a key-value structure that remains immune to character-based delimiters within the data fields.

In [19]:
import json

# Define the path to the extracted JSON file
json_path = os.path.join("resources", "games.json")

# Load the raw JSON file
# Using utf-8 encoding to ensure special characters in game titles are preserved
with open(json_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Convert the dictionary-based JSON to a Pandas DataFrame
# The JSON keys are AppIDs. Use orient='index' to map them correctly
df = pd.DataFrame.from_dict(raw_data, orient='index')

# Reset the index to move the AppID from the index to a proper column
df = df.reset_index().rename(columns={'index': 'AppID'})

# Standardize the Release Date
# Convert the string dates into datetime objects to enable sorting and filtering by year
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

# Preview the aligned data
print(f"Dataset successfully loaded. Total records: {len(df)}")
df.head()

Dataset successfully loaded. Total records: 122611


,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,2023-08-01,0,0.00,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,2025-04-24,0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


## Data Verification: Structural Integrity Restored

Now that the data is sourced from the JSON file, the structural misalignment is resolved. Each field, from game titles to review counts, is correctly mapped to its corresponding key. 

To confirm the accuracy of this fix, I will now inspect a few key titles that will be central to the **RPG Renaissance** analysis. This step ensures that metrics like `positive` reviews and `playtime` are no longer null or zero, providing a solid foundation for the upcoming SQL queries.

In [20]:
# List of some interesting titles to verify data integrity
verification_list = ['Cyberpunk 2077', 'The Witcher 3: Wild Hunt', 'ELDEN RING', 'Baldur\'s Gate 3',
                     'Tainted Grail: The Fall of Avalon', 'Pathfinder: Wrath of the Righteous - Enhanced Edition', 'Warhammer 40,000: Rogue Trader', 'Clair Obscur: Expedition 33']

# Filtering the fixed dataframe for these specific titles
check_data = df[df['name'].isin(verification_list)]

# Selecting key columns to confirm the fix
verification_view = check_data[['AppID', 'name', 'release_date', 'positive', 'negative', 
    'average_playtime_forever', 'median_playtime_forever']]

print("Verification of Key Titles:")
display(verification_view)

Verification of Key Titles:


,AppID,name,release_date,positive,negative,average_playtime_forever,median_playtime_forever
28679,2186680,"Warhammer 40,000: Rogue Trader",2023-12-07,26360,4445,3653,1334
39800,292030,The Witcher 3: Wild Hunt,2015-05-18,802993,32356,3471,875
48641,1903340,Clair Obscur: Expedition 33,2025-04-24,106216,4816,2168,1763
59354,1086940,Baldur's Gate 3,2023-08-03,736688,23974,7562,4698
68900,1091500,Cyberpunk 2077,2020-12-09,713071,131850,4961,3213
83741,1184370,Pathfinder: Wrath of the Righteous - Enhanced ...,2021-09-02,32303,6193,4438,753
105308,1466060,Tainted Grail: The Fall of Avalon,2025-05-23,10043,1437,1466,999
114885,1245620,ELDEN RING,2022-02-24,981540,75137,8155,5832


## Isolating RPG Titles and Calculating Quality Metrics

To focus the analysis on the RPG Renaissance, the dataset must be filtered to include only games belonging to the Role-Playing Game genre. Steam's metadata categorizes games using both 'genres' and 'tags'. By scanning both fields for the 'RPG' keyword, we ensure that sub-genres like Action-RPGs or CRPGs are included.

In this step, I am also introducing two critical metrics for the portfolio:
1. Total Reviews: The sum of positive and negative feedback, serving as a proxy for the game's popularity.
2. Score: The percentage of positive reviews, providing a standardized quality rating.

In [21]:
# Create a dedicated filter for RPGs
# Checking both 'genres' and 'tags' columns for the 'RPG' string
rpg_mask = (df['genres'].astype(str).str.contains('RPG', case=False, na=False)) | \
           (df['tags'].astype(str).str.contains('RPG', case=False, na=False))

# Create the new RPG-only DataFrame
df_rpg = df[rpg_mask].copy()

# Calculate metrics: Total Reviews and Score (Percentage of Positive)
df_rpg['total_reviews'] = df_rpg['positive'] + df_rpg['negative']

# Calculating the score as a percentage (0-100)
# We handle division by zero using .fillna(0)
df_rpg['score'] = (df_rpg['positive'] / df_rpg['total_reviews'] * 100).round(2).fillna(0)

# Filter out games with less than 300 reviews to keep the data relevant for analysis.
# Also filter out F2P titles as they may skew statistics
df_rpg = df_rpg[(df_rpg['total_reviews'] > 300) & (df_rpg['price'] > 0)]

# Selecting only essential columns for the upcoming Renaissance analysis
essential_columns = [
    'AppID', 'name', 'release_date', 'price', 'positive', 'negative', 
    'total_reviews', 'score', 'average_playtime_forever', 'median_playtime_forever'
]
df_rpg = df_rpg[essential_columns]

# Display top 10 RPGs by popularity to verify
print(f"Number of RPG titles isolated: {len(df_rpg)}")
display(df_rpg.sort_values(by='total_reviews', ascending=False).head(10))

Number of RPG titles isolated: 3759


,AppID,name,release_date,price,positive,negative,total_reviews,score,average_playtime_forever,median_playtime_forever
10927,105600,Terraria,2011-05-16,4.99,1373979,35494,1409473,97.48,8245,2051
5377,252490,Rust,2018-02-08,19.99,1071135,156649,1227784,87.24,24264,3044
44134,2358720,Black Myth: Wukong,2024-08-19,59.99,1111720,38378,1150098,96.66,3272,2812
114885,1245620,ELDEN RING,2022-02-24,38.99,981540,75137,1056677,92.89,8155,5832
95880,413150,Stardew Valley,2016-02-26,8.99,872384,13811,886195,98.44,4830,1728
68900,1091500,Cyberpunk 2077,2020-12-09,20.99,713071,131850,844921,84.39,4961,3213
39800,292030,The Witcher 3: Wild Hunt,2015-05-18,3.99,802993,32356,835349,96.13,3471,875
59354,1086940,Baldur's Gate 3,2023-08-03,44.99,736688,23974,760662,96.85,7562,4698
14153,346110,ARK: Survival Evolved,2017-08-27,9.89,612177,117993,730170,83.84,11685,921
3692,218620,PAYDAY 2,2013-08-13,0.99,594327,68755,663082,89.63,7427,546


## Defining the Eras: Golden Age vs. Modern Renaissance

With a clean and filtered dataset of **3759** RPG titles, the analysis now moves toward comparing two pivotal periods in gaming history. By segmenting the data based on release years, I will evaluate how the genre has evolved in terms of player engagement and overall quality.

The next phase of this project focuses on:
* **The Golden Age vs. Renaissance:** Comparing the traditional "Golden Era" of RPGs against the modern surge of high-quality releases post-2020.
* **Playtime Evolution:** Investigating whether modern RPGs have successfully increased player retention (Average/Median Playtime) compared to historical benchmarks.
* **Engagement Metrics:** Analyzing if the "Renaissance" titles command higher scores and larger player bases than their predecessors.

In [22]:
# Defining the time periods for comparison
golden_age = df_rpg[df_rpg['release_date'].dt.year <= 2010].copy()
modern_renaissance = df_rpg[df_rpg['release_date'].dt.year >= 2020].copy()

# Calculating key metrics for both eras
def calculate_era_stats(df_era, name):
    stats = {
        'Era': name,
        'Game Count': len(df_era),
        'Avg Score': df_era['score'].mean().round(2),
        'Median Score': df_era['score'].median().round(2),
        'Avg Median Playtime (Hours)': (df_era['median_playtime_forever'].mean() / 60).round(2),
        'Max Median Playtime (Hours)': (df_era['median_playtime_forever'].max() / 60).round(2)
    }
    return stats

# Creating a summary table
era_comparison = pd.DataFrame([
    calculate_era_stats(golden_age, "Golden Age (<= 2010)"),
    calculate_era_stats(modern_renaissance, "Modern Renaissance (>= 2020)")
])

print("Statistical Comparison of RPG Eras:")
display(era_comparison)

Statistical Comparison of RPG Eras:


,Era,Game Count,Avg Score,Median Score,Avg Median Playtime (Hours),Max Median Playtime (Hours)
0,Golden Age (<= 2010),117,82.21,85.44,5.35,37.20
1,Modern Renaissance (>= 2020),2122,82.27,84.94,19.86,5383.05


## Statistical Insights: The Shift in RPG Engagement

The initial comparison between the **Golden Age** and the **Modern Renaissance** reveals a significant evolution in the RPG landscape. While the Golden Age established the foundations of the genre with highly-rated classics, the Modern Renaissance demonstrates a massive surge in both the volume of high-quality releases and player commitment.

Key observations from the data:
* **Quality Consistency:** The average and median scores remain remarkably high in the modern era, suggesting that the increase in production volume has not diluted the overall quality of the genre.
* **Player Retention:** Modern RPGs show a noticeable trend in playtime metrics, reflecting the shift towards more expansive, "live-service," or highly replayable open-world structures.
* **Market Volume:** The number of significant RPG titles (with >300 reviews) released since 2020 is rapidly approaching or exceeding the cumulative totals of previous decades, highlighting the genre's current dominance on Steam.

In [23]:
# Creating a labeling function for Eras
def assign_era(release_date):
    year = release_date.year
    if year <= 2010:
        return 'Golden Age'
    elif year >= 2020:
        return 'Modern Renaissance'
    else:
        return 'Intermediate Period'

# Applying the labels to our RPG dataset
df_rpg['era'] = df_rpg['release_date'].apply(assign_era)

# Final export to CSV for Tableau
# We save it without the index to keep the file clean
df_rpg.to_csv('rpg_renaissance_clean.csv', index=False)

print("File 'rpg_renaissance_clean.csv' is ready for Tableau!")
print(f"Final row count: {len(df_rpg)}")
display(df_rpg.head())

File 'rpg_renaissance_clean.csv' is ready for Tableau!
Final row count: 3759


,AppID,name,release_date,price,positive,negative,total_reviews,score,average_playtime_forever,median_playtime_forever,era
42,416790,The Metronomicon: Slay The Dance Floor,2016-09-29,4.99,330,65,395,83.54,183,237,Intermediate Period
65,715560,Eastshade,2019-02-13,9.99,4608,549,5157,89.35,320,151,Intermediate Period
69,375510,New kind of adventure,2015-06-04,0.99,632,881,1513,41.77,217,212,Intermediate Period
84,2997980,Natsu no Sagashimono ~What We Found That Summer~,2024-09-27,11.99,687,92,779,88.19,335,402,Modern Renaissance
95,914710,Cat Quest II,2019-09-24,3.74,6092,273,6365,95.71,263,159,Intermediate Period


## Preparing for Tableau Visualization

The data processing phase is now complete. By finalizing the dataset in Python, I have ensured that the visual analysis in Tableau will be performant and logically structured. 

Key enhancements made for the visualization stage:
* **Era Categorization:** A new categorical column has been added to instantly distinguish between the Golden Age and the Modern Renaissance.
* **Metric Pre-calculation:** Engagement scores and review totals are pre-calculated to avoid complex formulas within the BI tool.
* **Optimized File Size:** The dataset has been reduced from an 800MB raw JSON to a highly focused, several-thousand-row CSV, ready for immediate dashboarding.